# Procesamiento Experimental (video l4.mp4)

Este cuaderno procesa el video del cohete y construye su trayectoria.

**Datos del ensayo:** botella de 1.5 L, fracción de agua φ = 1/3 (0.5 L), inflada con bombín de bicicleta (la presión no quedó registrada).

**Geometría del montaje:** lanzador de 1.71 m; cámara a 18 m de distancia, sobre un trípode de 1.71 m.

**Antes de ejecutar:** sube `l4.mp4` a la misma carpeta que este cuaderno (en Colab: panel de archivos de la izquierda).

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### 1.1 Lectura del video

In [ ]:
VIDEO = "l4.mp4"

cap = cv2.VideoCapture(VIDEO)
FPS = cap.get(cv2.CAP_PROP_FPS)
ANCHO = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
ALTO = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"Cuadros por segundo: {FPS:.3f}")
print(f"Resolucion: {ANCHO} x {ALTO} pixeles")


def obtener_frame(t):
    """Devuelve el cuadro del video en el instante t (s) como imagen RGB."""
    cap = cv2.VideoCapture(VIDEO)
    cap.set(cv2.CAP_PROP_POS_FRAMES, round(t * FPS))
    ok, frame_bgr = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f"No se pudo leer el video en t = {t} s")
    return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

### 1.1 Explorador de cuadros

Revisión cuadro por cuadro.

In [ ]:
def ver_cuadros(t_inicio, t_fin, paso=0.1, columnas=5, con_grilla=True):
    """Muestra en una grilla los cuadros entre t_inicio y t_fin, uno cada 'paso' segundos."""
    tiempos = np.arange(t_inicio, t_fin, paso)
    filas = int(np.ceil(len(tiempos) / columnas))
    fig, axs = plt.subplots(filas, columnas, figsize=(3 * columnas, 5.3 * filas))
    axs = np.array(axs).reshape(-1)

    for ax, t in zip(axs, tiempos):
        ax.imshow(obtener_frame(t))
        if con_grilla:
            for x in range(0, ANCHO, 100):
                ax.axvline(x, color="red", lw=0.3, alpha=0.5)
            for y in range(0, ALTO, 100):
                ax.axhline(y, color="lime", lw=0.3, alpha=0.5)
        ax.set_title(f"t = {t:.3f} s", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])

    for ax in axs[len(tiempos):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

ver_cuadros(17, 20, paso=1 / FPS)

### 1.2 Calibración de escala: altura de una persona

El operador, de pie a los t = 2.0 s, mide 1.71 m.

In [ ]:
t_persona = 2.0
frame_persona = obtener_frame(t_persona)

cabeza_px = 481   # fila del tope del casco
pies_px = 545     # fila donde el zapato toca el pasto
altura_px = pies_px - cabeza_px
altura_m = 1.71

escala = altura_m / altura_px   # metros por pixel
print(f"Altura de la persona: {altura_px} px")
print(f"Escala: {escala:.5f} m/px")

fig, ax = plt.subplots(figsize=(3.5, 5))
ax.imshow(frame_persona)
ax.axhline(cabeza_px, color="red", lw=1.5, label="cabeza")
ax.axhline(pies_px, color="lime", lw=1.5, label="pies")
ax.set_xlim(120, 200)
ax.set_ylim(570, 450)
ax.legend(fontsize=8, loc="upper right")
ax.set_title(f"Calibracion de escala (t = {t_persona} s)\naltura = {altura_px} px")
plt.tight_layout()
plt.show()

### 1.3 Posiciones del cohete (medidas a mano y detección de control)

In [ ]:
GANANCIA = 2.5   # video nocturno: se sube el brillo (tambien en la verificacion visual)


def candidatos_blancos(frame_rgb, s_max=60, v_min=175):
    """Manchas blancas o brillantes y compactas del cuadro.

    Devuelve una lista de (columna, fila, area).
    """
    brillante = np.clip(frame_rgb.astype(float) * GANANCIA, 0, 255).astype(np.uint8)
    hsv = cv2.cvtColor(brillante, cv2.COLOR_RGB2HSV)
    mascara = ((hsv[:, :, 1] < s_max) & (hsv[:, :, 2] >= v_min)).astype(np.uint8) * 255

    n_manchas, _, stats, centroides = cv2.connectedComponentsWithStats(mascara, connectivity=8)
    candidatos = []
    for k in range(1, n_manchas):
        x, y, w, h, area = stats[k]
        if area < 4:
            continue
        if area / (w * h) < 0.15:   # descarta manchas muy ralas
            continue
        candidatos.append((centroides[k][0], centroides[k][1], area))
    return candidatos


def trackear(cuadros, semilla_xy, semilla_v, s_max=60, v_min=175):
    """Sigue el cohete cuadro a cuadro: elige la mancha mas cercana a la posicion predicha.

    semilla_xy: (columna, fila) en el primer cuadro (px); semilla_v: desplazamiento por cuadro (px).
    Si un cuadro no tiene candidatos, se mantiene la posicion predicha.
    Devuelve (columnas, filas, cuadros sin deteccion).
    """
    x, y = semilla_xy
    vx, vy = semilla_v
    columnas, filas, sin_deteccion = [x], [y], 0

    for k in cuadros[1:]:
        candidatos = candidatos_blancos(obtener_frame(k / FPS), s_max, v_min)
        x_pred, y_pred = x + vx, y + vy
        if candidatos:
            mejor = min(candidatos, key=lambda c: (c[0] - x_pred) ** 2 + (c[1] - y_pred) ** 2)
            x_nuevo, y_nuevo = mejor[0], mejor[1]
        else:
            x_nuevo, y_nuevo = x_pred, y_pred
            sin_deteccion += 1
        vx, vy = x_nuevo - x, y_nuevo - y
        x, y = x_nuevo, y_nuevo
        columnas.append(x)
        filas.append(y)

    return np.array(columnas), np.array(filas), sin_deteccion

In [ ]:
# (cuadro, columna_px, fila_px, dx_camara_px, dy_camara_px)
# dy_camara: cuanto bajo la escena respecto al cuadro de calibracion (la camara estaba en la mano)
# El video repite cuadros (446/447, 451/452, 457/458, 472/473): se deja solo el primero de cada pareja.
# Los cuadros intermedios (439-441, 443-445, ...) se interpolan; el despegue (~430-437) y la caida final (>473) no se pudieron seguir.
datos = [
    (438,  73.0, 210.0, -42.6, -49.8),
    (442,  58.0, 165.0, -42.5, -51.2),
    (446,  48.0, 135.0, -43.3, -52.8),
    (451,  41.0, 125.0, -43.4, -54.7),   # cerca del punto mas alto
    (457,  37.0, 135.0, -43.6, -56.0),   # camara corregida (la correlacion dio un valor atipico)
    (464,  35.0, 158.0, -44.2, -59.0),
    (472,  41.0, 211.0, -44.3, -61.8),
]

D = np.array(datos, float)
k_dat = D[:, 0]

cuadros = np.arange(int(k_dat[0]), int(k_dat[-1]) + 1)   # cuadros 438 a 472 (~18.3 a 19.8 s)
t_visibles = cuadros / FPS

col_px = np.interp(cuadros, k_dat, D[:, 1])
fila_px = np.interp(cuadros, k_dat, D[:, 2])
dy_cam = np.interp(cuadros, k_dat, D[:, 4])

# True = medido; False = interpolado
medido = np.isin(cuadros, k_dat)

In [ ]:
# Detector aplicado a los cuadros desde el primer al ultimo dato de la tabla.
# La posicion y el desplazamiento iniciales salen de los dos primeros puntos.
def imprimir_datos(cuadros, col_px, fila_px, dx_cam, dy_cam, por_linea=2):
    "Imprime las posiciones en el formato de la lista 'datos'."
    filas = [f"({k}, {c:5.1f}, {f:5.1f}, {dx:5.1f}, {dy:5.1f})" for k, c, f, dx, dy in zip(cuadros, col_px, fila_px, dx_cam, dy_cam)]
    print("datos = [")
    for i in range(0, len(filas), por_linea):
        print("    " + ", ".join(filas[i:i + por_linea]) + ",")
    print("]")


cuadros_det = cuadros[cuadros <= k_dat[-1]]
col_det, fila_det, sin_det = trackear(cuadros_det, (col_px[0], fila_px[0]),
                                      (col_px[1] - col_px[0], fila_px[1] - fila_px[0]))

en_tabla = np.isin(cuadros_det, k_dat)   # se compara solo en los cuadros de la tabla
imprimir_datos(cuadros_det[en_tabla], col_det[en_tabla], fila_det[en_tabla], D[:, 3], D[:, 4])
print(f"\nDiferencia maxima con 'datos': {np.abs(col_det[en_tabla] - D[:, 1]).max():.1f} px en columna, "
      f"{np.abs(fila_det[en_tabla] - D[:, 2]).max():.1f} px en fila")
print(f"Cuadros sin deteccion: {sin_det} de {len(cuadros_det)}")

**Verificación visual.**

In [ ]:
R = 30       # semilado de la ventana (px)
P = 2 * R    # relleno: permite puntos hasta R px fuera de la imagen
ncols = 6
nrows = int(np.ceil(len(cuadros) / ncols))
fig, axs = plt.subplots(nrows, ncols, figsize=(2.6 * ncols, 2.6 * nrows))
axs = np.atleast_1d(axs).reshape(-1)

for ax, k, t, xc, yc, ok in zip(axs, cuadros, t_visibles, col_px, fila_px, medido):
    padded = np.pad(obtener_frame(t), ((P, P), (P, P), (0, 0)), mode="constant")
    x0, y0 = int(round(xc)), int(round(yc))
    recorte = padded[y0 - R + P:y0 + R + P, x0 - R + P:x0 + R + P]
    recorte = np.clip(recorte.astype(float) * GANANCIA, 0, 255).astype(np.uint8)   # video nocturno: se sube el brillo
    ax.imshow(recorte, extent=[x0 - R, x0 + R, y0 + R, y0 - R])
    ax.plot(xc, yc, "o", mfc="none", mec="red" if ok else "yellow", ms=14, mew=2)   # rojo = medido, amarillo = no medido
    ax.set_title(f"k = {k}  t = {t:.3f} s" + ("" if ok else " (interp.)"), fontsize=8)
    ax.axis("off")
for ax in axs[len(cuadros):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

### 1.4 Pasando a unidades físicas

In [ ]:
t_rel = t_visibles - t_visibles[0]
n = len(t_rel)

# Altura con escala constante (aproximacion), corrigiendo el movimiento de la camara
y_lineal = (pies_px - (fila_px - dy_cam)) * escala

# Altura con correccion de perspectiva (camara a D_cam m de distancia, sobre tripode de h_cam m)
D_cam, h_cam = 18.0, 1.71                 # (m)
f_px = D_cam / escala                    # distancia focal equivalente (px)
cy = ALTO / 2                            # fila central de la imagen
inclinacion0 = np.arctan((pies_px - cy) / f_px) - np.arctan(h_cam / D_cam)   # inclinacion de la camara en la calibracion
elevacion = inclinacion0 + np.arctan((cy - (fila_px - dy_cam)) / f_px)
y_m = h_cam + D_cam * np.tan(elevacion)  # altura sobre el suelo (m)
x_m = col_px * escala

print(f"Inclinacion de la camara en la calibracion: {np.degrees(inclinacion0):.1f} grados")
print(f"Distancia focal equivalente: {f_px:.0f} px\n")


def derivada(t, x):
    """Velocidad por diferencias centrales (en los extremos, diferencias simples)."""
    v = np.zeros(len(t))
    v[1:-1] = (x[2:] - x[:-2]) / (t[2:] - t[:-2])
    v[0] = (x[1] - x[0]) / (t[1] - t[0])
    v[-1] = (x[-1] - x[-2]) / (t[-1] - t[-2])
    return v


vx = derivada(t_rel, x_m)
vy = derivada(t_rel, y_m)

# Aceleracion vertical por segunda diferencia (sin valor en los extremos)
ay = np.full(n, np.nan)
ay[1:-1] = (y_m[2:] - 2 * y_m[1:-1] + y_m[:-2]) / (1 / FPS) ** 2

print(pd.DataFrame({"t (s)": t_rel.round(3), "x (m)": x_m.round(2), "y (m)": y_m.round(2),
                    "vx (m/s)": vx.round(2), "vy (m/s)": vy.round(2), "ay (m/s2)": ay.round(1)}).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].plot(t_rel, y_m, "o-")
ax[0].set_ylabel("y (m)")
ax[0].set_title("Altura")
ax[1].plot(t_rel, vy, "o-")
ax[1].set_ylabel("vy (m/s)")
ax[1].set_title("Velocidad vertical")
for a in ax:
    a.set_xlabel("t (s)")
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 1.5 Exportar a Excel

In [ ]:
df_exp = pd.DataFrame({
    "t": np.round(t_rel, 4),
    "y": np.round(y_m, 3),
    "y_lineal": np.round(y_lineal, 3),
    "vy": np.round(vy, 2),
    "ay": np.round(ay, 1),
    "step": np.arange(n),
    "tramo": 1,
})
df_exp.to_excel("Dato_exp_l4.xlsx", index=False)
df_exp